## Preparation of the data

### 1. Custom dataset class

In [21]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from typing import Optional, Dict, Any, Callable, List, Union, Tuple

class CustomEnricoDataset(Dataset):
    def __init__(self, 
                 root: str,
                 screen_ids: List[str],
                 labels_dict: Dict[str, str],
                 class_to_idx: Dict[str, int],
                 transform: Optional[Callable] = None, 
                 transform_to_class: Optional[Dict[str, Union[Callable, List[Callable]]]] = None):
        """
        Internal constructor. Use CustomEnricoDataset.create_splits() to initialize 
        train and test datasets comfortably from a Kaggle path.
        """
        self.root = root
        self.img_dir = os.path.join(self.root, "screenshots/screenshots")
        self.transform = transform
        self.transform_to_class = transform_to_class or {}
        
        self.screen_ids = screen_ids
        self.labels_dict = labels_dict
        self.class_to_idx = class_to_idx

        # Build the Flat Index Map (Augmentation Expansion)
        self.samples = []
        for screen_id in self.screen_ids:
            label_str = self.labels_dict.get(screen_id)
            if label_str:
                # Store (screen_id, specific_transform, is_augmented_flag)
                self.samples.append((screen_id, None)) # Original image
                
                if label_str in self.transform_to_class:
                    augs = self.transform_to_class[label_str]
                    augs = [augs] if not isinstance(augs, list) else augs
                    for aug in augs:
                        self.samples.append((screen_id, aug)) # Augmented copy

    @classmethod
    def create_splits(cls, 
                      root: str, 
                      test_size: float = 0.2, 
                      seed: int = 42,
                      transform: Optional[Callable] = None,
                      transform_to_class: Optional[Dict] = None) -> Tuple['CustomEnricoDataset', 'CustomEnricoDataset']:
        """
        Reads the Kaggle directory, performs a stratified train/test split,
        and returns both datasets configured automatically.
        """
        csv_path = os.path.join(root, "design_topics.csv")
        
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"Could not find {csv_path}. Check your Kaggle input path.")

        # 1. Load Metadata
        df = pd.read_csv(csv_path)
        df['screen_id'] = df['screen_id'].astype(str)

        # 2. Perform Stratified Split
        # stratify=df['topic'] ensures labels are proportionally represented in train/test
        train_df, test_df = train_test_split(
            df, 
            test_size=test_size, 
            stratify=df['topic'], 
            random_state=seed
        )

        # 3. Build shared dictionaries for consistent label encoding
        labels_dict = dict(zip(df['screen_id'], df['topic']))
        unique_labels = sorted(df['topic'].unique())
        class_to_idx = {label: idx for idx, label in enumerate(unique_labels)}

        # 4. Create Train Dataset (Includes specific augmentations)
        train_dataset = cls(
            root=root,
            screen_ids=train_df['screen_id'].tolist(),
            labels_dict=labels_dict,
            class_to_idx=class_to_idx,
            transform=transform,
            transform_to_class=transform_to_class
        )

        # 5. Create Test Dataset (No class augmentations - testing should be clean data)
        test_dataset = cls(
            root=root,
            screen_ids=test_df['screen_id'].tolist(),
            labels_dict=labels_dict,
            class_to_idx=class_to_idx,
            transform=transform,
            transform_to_class=transform_to_class 
        )

        return train_dataset, test_dataset

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        screen_id, specific_transform = self.samples[index]
        img_path = os.path.join(self.img_dir, f"{screen_id}.jpg") 
        
        image = Image.open(img_path)
        
        # Apply specific augmentation (PIL level)
        if specific_transform: 
            image = specific_transform(image)
            
        # Apply base transforms (Tensor level)
        if self.transform: 
            image = self.transform(image)

        label_idx = self.class_to_idx[self.labels_dict[screen_id]]
        return image, label_idx

    def get_by_label(self, label: str, num_samples: int = 5):
        """Utility to retrieve images belonging to a specific label."""
        results = []
        for i, (screen_id, aug) in enumerate(self.samples):
            if self.labels_dict[screen_id] == label:
                print(f"Label={label} proc")
                img, _ = self.__getitem__(i)

                results.append(img)
                
                if len(results) > num_samples:
                    break
        return results

### 2. Defining the transformations and per class transformations for the classes that are underrepresented

In [22]:
from torchvision.transforms import v2
non_identity_perms = [
    [0, 2, 1], # R-B-G
    [1, 0, 2], # G-R-B
    [1, 2, 0], # G-B-R
    [2, 0, 1], # B-R-G
    [2, 1, 0]  # B-G-R
]

forced_channel_shuffle = v2.Lambda(
    lambda x: x[random.choice(non_identity_perms), :, :]
)
# 1. Aggressive Augmentations for Classes with < 20 samples
rare_class_aug = v2.Compose([
    v2.RandomPerspective(distortion_scale=0.2, p=0.5),
    v2.RandomAffine(degrees=5, translate=(0.1, 0.1)),
    v2.ColorJitter(brightness=0.3, contrast=0.3),
    v2.RandomGrayscale(p=0.2)
])

# 2. Physiological Fatigue for Stage 4 (CVS focus)
fatigue_aug = v2.Compose([
    v2.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    v2.ColorJitter(brightness=0.5) # Simulating PWM Flicker
])

# 3. Map to your dictionary
augmentations = {
    "dialer": [rare_class_aug, v2.RandomHorizontalFlip(p=0.5)],
    "camera": [rare_class_aug, v2.RandomAdjustSharpness(sharpness_factor=2)],
    "maps": [rare_class_aug, v2.RandomInvert(p=0.2)],
    "tutorial": [rare_class_aug] # Also rare, so needs boost
}

universal_base = v2.Compose([
    v2.RGB(),
    v2.ToImage(),
    v2.ToDtype(torch.uint8, scale=True),
    v2.Resize(size=(320, 180)),
    v2.ToDtype(torch.float32, scale=True),
    
])

### 3. Generating training and test dataset

In [23]:
train_dataset, test_dataset = CustomEnricoDataset.create_splits(
    root="/kaggle/input/datasets/nazariyyuchnovskiy/enrico-dataset-vanilla", 
    test_size=0.2, # 20% goes to Test
    seed=317,       # Keeps the split identical across runs
    transform=universal_base,
    transform_to_class=augmentations
)

**Additional functions to run the model**

In [24]:
#checkpointing logic
import os
from typing import Optional

def save_checkpoint(model, optimizer, epoch, loss, path="/kaggle/working/", file_name: Optional[str] = None):
    """
    Saves the current state of training per epoch. but willl be practically done per each 10 epoch due to memory contraints.
    """
    state = {
        'epoch': epoch,
        'state_dict': model.state_dict(),     # The actual weights
        'optimizer': optimizer.state_dict(),   # Momentum, learning rates, etc.
        'loss': loss,
    }
    if file_name is None:
        torch.save(state, f"{path}/after_{epoch}_checkpoint.pth")
    else:
        torch.save(state, f"{path}{file_name}.pth")
    print(f"Checkpoint saved to {path} at epoch {epoch}")

def load_checkpoint(model, optimizer: Optional[None], path ="/kaggle/input/"):
    """
    Loads a saved state and updates the model and optimizer.
    """
    if os.path.exists(f'{path}'):
        checkpoint = torch.load(path)
        load_status = model.load_state_dict(checkpoint['state_dict'])
        print(f'The load_status is {load_status}')
        if optimizer is not None:
            optimizer.load_state_dict(checkpoint['optimizer'])
        epoch = checkpoint['epoch']
        loss = checkpoint['loss']
        print(f"Reverting the model from epoch {epoch} with loss {loss:.4f}")
        return epoch
    else:
        print("No checkpoint found at this path.")
        return 0

In [26]:
!pip install torch_xla
import torch_xla

device = torch_xla.device()
batch_size = 128# since we are using v5e-1 TPU, have 64 as the best option, including the fact that images are tiff, which are much higher quality than jpg
screen_train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
screen_test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

class Args:
    epochs = 50            # Increased to 50 so you can actually see the checkpointing trigger
    learning_rate = 0.01
    momentum = 0.9
    seed = 317
    log_interval = 3


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Preparation of the model

### 1. Model itself

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

def init_weights(m):
    # Check if the sub-module is a Linear layer
    if isinstance(m, nn.Linear):
        # Reach inside the module and grab the 'weight' tensor
        nn.init.xavier_uniform_(m.weight)
        # Often we initialize biases to a small constant or zero
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.01)
            
    # Check if it's a Convolutional layer
    elif isinstance(m, nn.Conv2d):
        nn.init.xavier_uniform_(m.weight)
        
class PseudoAlexNet(nn.Module):
    def __init__(self):
        super(PseudoAlexNet, self).__init__()
        # Input: 3 x 300 x 200
        
        # Conv 1
        self.conv1 = nn.Conv2d(3, 96, kernel_size=7, stride=3, padding=3)
        self.bn1 = nn.BatchNorm2d(96)
        
        # Conv 2
        self.conv2 = nn.Conv2d(96, 128, kernel_size=5, stride=2, padding=2)
        self.bn2 = nn.BatchNorm2d(128)
        
        # Conv 3
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        
        # Conv 4
        self.conv4 = nn.Conv2d(256, 384, kernel_size=3, stride=1, padding=1)
        self.bn4 = nn.BatchNorm2d(384)
        
        # Linear Layers (Flattened size: 384 * 12 * 8 = 36864)
        self.fc1 = nn.Linear(384 * 12 * 8, 256)
        self.bn_fc = nn.BatchNorm1d(256) # 1D for Linear layers
        self.fc2 = nn.Linear(256, 101)

    def forward(self, x):
        # Layer 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        
        # Layer 2 + Pool
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2, stride=2)

        # Layer 3
        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        
        # Layer 4 + Pool
        x = self.conv4(x)
        x = self.bn4(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2, stride=2)
        
        # Flatten
        x = x.view(x.size(0), -1) 
        
        # Fully Connected 1
        x = self.fc1(x)
        x = self.bn_fc(x)
        x = F.relu(x)
        
        # Output
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

### 2. Loading the previous weights

In [57]:
checkpoint_path = '/kaggle/input/datasets/nazariyyuchnovskiy/20-checkpoint-batchnorm/after_20_checkpoint (1).pth'
model = PseudoAlexNet()

model.to(device)

load_checkpoint(model=model, optimizer=None, path=checkpoint_path)

The load_status is <All keys matched successfully>
Reverting the model from epoch 20 with loss 0.0122


20

### 3. Functions for the proper fine_tuning

In [58]:
def initial_freeze_unfreeze(model):
    for param in model.parameters():
        param.requires_grad = False
    
    for param in model.fc1.parameters():
        param.requires_grad = True
    
    for param in model.bn_fc.parameters():
        param.requires_grad = True


def second_unfreeze(model):
    for param in model.conv4.parameters():
        param.requires_grad = True
        
    for param in model.conv3.parameters():
        param.requires_grad = True


fc_last_in_features = model.fc2.in_features
model.fc1 = nn.Linear(in_features=384*13*7, out_features=fc_last_in_features)
model.fc2 = nn.Linear(in_features=fc_last_in_features, out_features=16) # exactly so much classes are in RVL-CDIP

## Fine-tuning

### 1. Training and eval epoch function

In [59]:
import torch.nn.functional as F
import time
import torch
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr
import torch_xla

from typing import List, Tuple, Optional

def train_and_eval_epoch(epoch, model, mp_train_loader, mp_test_loader, 
                         optimizer, loss_fn, device, args, warmup_epoch: bool = False) -> Tuple[float, Optional[float]]:
    epoch_start_time = time.time()

    # ==========================================
    # --- TRAINING LOOP ---
    # ==========================================
    model.train()
    local_train_loss_sum = 0.0
    local_train_steps = 0

    for batch_idx, (data, target) in enumerate(mp_train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = loss_fn(output, target)
        loss.backward()

        xm.optimizer_step(optimizer)
        
        # Accumulate local loss for epoch averaging
        local_train_loss_sum += loss.item()
        local_train_steps += 1

        # --- LOGGING ---
        if batch_idx % args.log_interval == 0:
            if xm.is_master_ordinal(local=False):
                print(f'Train Epoch: {epoch} '
                      f'[{batch_idx * len(data) * xr.world_size()}/{len(mp_train_loader)*128} '
                      f'({100. * batch_idx / len(mp_train_loader):.0f}%)]\tLoss: {loss.item():.6f}')

    metrics = torch.tensor([local_train_loss_sum, local_train_steps], dtype=torch.float32, device=device)
    
    # 2. Sum the tensor across all 8 TPU cores natively
    global_metrics = xm.all_reduce(xm.REDUCE_SUM, metrics)
    global_train_loss = global_metrics[0].item()
    global_train_steps = global_metrics[1].item()
    
    if global_train_steps > 0:
        avg_train_loss = global_train_loss / global_train_steps
    else:
        avg_train_loss = float('inf')
    
    avg_train_loss = global_train_loss / global_train_steps

    if xm.is_master_ordinal(local=False):
        print(f'--- Epoch {epoch} Training Metrics ---')
        print(f'Average Train Loss: {avg_train_loss:.6f}')


    # ==========================================
    # --- EVALUATION LOOP ---
    # ==========================================    
    model.eval()
    local_test_loss_sum = 0.0
    local_test_steps = 0

    with torch.no_grad():
        for data, target in mp_test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = loss_fn(output, target)
            
            local_test_loss_sum += loss.item()
            local_test_steps += 1

    # Reduce test metrics across all TPU cores
    metrics = torch.tensor([local_test_loss_sum, local_test_steps], dtype=torch.float32, device=device)
    
    # 2. Sum the tensor across all 8 TPU cores natively
    global_metrics = xm.all_reduce(xm.REDUCE_SUM, metrics)
    global_test_loss = global_metrics[0].item()
    global_test_steps = global_metrics[1].item()
    
    avg_test_loss = global_test_loss / global_test_steps
    
    if epoch %5 == 0 and xm.is_master_ordinal(local=False):
        print(f'\n--- Epoch {epoch} Evaluation ---')
        print(f'Test set: Average Loss: {avg_test_loss:.6f}\n')

    # ==========================================
    # --- CHECKPOINTING & TIMING ---
    # ==========================================
    if epoch % 10 == 0:
        # We restrict the save function to the master ordinal to prevent file corruption
        if xm.is_master_ordinal(local=False):
            file_name = f"checkpoint_after_{epoch}_warmup" if warmup_epoch else f"checkpoint_after_{epoch}_drift"
            save_checkpoint(model, optimizer, epoch, avg_train_loss, file_name=file_name)

    epoch_finish_time = time.time()

    if xm.is_master_ordinal(local=False):
        print(f"Total time of epoch {epoch} is {epoch_finish_time - epoch_start_time:.2f}s\n")

    # Return the losses so the Coordinator can use them
    return avg_train_loss, avg_test_loss

### 2. Preparation of the parameters for fine-tuning

In [60]:
import torch.optim
import torch_xla.debug.metrics as met
import torch_xla.distributed.parallel_loader as pl

MAX_WARMUP_EPOCHS = 20
MAX_DRIFT_EPOCHS = 30
CONFIDENCE_THRESHOLD = 0.01
ACC_DELTA_THRESHOLD = 0.005
SEED = 317


#defining variables needed for the whole double-stage training cycle and setting up the rest.
torch.manual_seed(SEED)
device = torch_xla.device()
model.to(device)

if xm.is_master_ordinal(local=False):
        print(f"Starting distributed training on {xr.world_size()} XLA devices...")
    
loss_fn = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-5, momentum=0.9)

mp_train_loader = pl.MpDeviceLoader(screen_train_loader, device)
mp_test_loader = pl.MpDeviceLoader(screen_test_loader, device)

test_loss_list_warmup = []
train_loss_list_warmup = []

test_loss_list_drift = []
train_loss_list_drift = []

args = Args()

Starting distributed training on 1 XLA devices...


### 3. Stage-1 of fine-tuning

In [61]:
print("=== STAGE 1: Head Warm-up ===")

prev_loss = 0.0
#initial freeze of all layers and unfreeze of the fully connected layers.
print(f"Starting training of the stage 1 in {MAX_WARMUP_EPOCHS} epochs.")
start_warmup_time = time.time()
initial_freeze_unfreeze(model=model)
for epoch in range(1, MAX_WARMUP_EPOCHS + 1):
    print(f"Warm-up Epoch {epoch}/{MAX_WARMUP_EPOCHS}...")
    
    # Run your function
    train_loss, test_loss = train_and_eval_epoch(epoch=epoch, model=model, mp_train_loader=mp_train_loader,
                                                       mp_test_loader=mp_test_loader,optimizer=optimizer, loss_fn=loss_fn,
                                                       device=device, args=args, warmup_epoch=True,)
    
    train_loss_list_warmup.append(train_loss)
    test_loss_list_warmup.append(test_loss)
    loss_diff = abs(prev_loss - test_loss)
    # THE BREAK CONDITION
    if loss_diff <= CONFIDENCE_THRESHOLD:
        print(f"\n>>> THRESHOLDS MET at Epoch {epoch} <<<")
        print(">>> Head is fully calibrated. Breaking warm-up cycle.")
        save_checkpoint(model=model, optimizer=optimizer, epoch=epoch, loss=test_loss_list_warmup[-1]) 
        break # Exit the warm-up loop early

    if epoch == MAX_WARMUP_EPOCHS:
        save_checkpoint(model=model, optimizer=optimizer, epoch=epoch, loss=test_loss_list_warmup[-1])       
    prev_loss = test_loss

print(f"Finished stage 1 in {time.time() - start_warmup_time}")

=== STAGE 1: Head Warm-up ===
Starting training of the stage 1 in 20 epochs.
Warm-up Epoch 1/20...
Train Epoch: 1 [0/1408 (0%)]	Loss: 2.859650
Train Epoch: 1 [384/1408 (27%)]	Loss: 2.866971
Train Epoch: 1 [768/1408 (55%)]	Loss: 2.941122
Train Epoch: 1 [1152/1408 (82%)]	Loss: 2.883251
--- Epoch 1 Training Metrics ---
Average Train Loss: 2.903751
Total time of epoch 1 is 26.00s

Warm-up Epoch 2/20...
Train Epoch: 2 [0/1408 (0%)]	Loss: 2.862926
Train Epoch: 2 [384/1408 (27%)]	Loss: 2.792403
Train Epoch: 2 [768/1408 (55%)]	Loss: 2.944207
Train Epoch: 2 [1152/1408 (82%)]	Loss: 2.771295
--- Epoch 2 Training Metrics ---
Average Train Loss: 2.850238
Total time of epoch 2 is 27.56s

Warm-up Epoch 3/20...
Train Epoch: 3 [0/1408 (0%)]	Loss: 2.814840
Train Epoch: 3 [384/1408 (27%)]	Loss: 2.840835
Train Epoch: 3 [768/1408 (55%)]	Loss: 2.793375
Train Epoch: 3 [1152/1408 (82%)]	Loss: 2.765684
--- Epoch 3 Training Metrics ---
Average Train Loss: 2.793611
Total time of epoch 3 is 26.79s

Warm-up Epoch 

### 4. Stage-2 of fine-tuning

In [62]:
#unfreezing two other convolutional layers
second_unfreeze(model=model)
# Drop the learning rate so the newly unfrozen layers don't shatter
for param_group in optimizer.param_groups:
    param_group['lr'] = 1e-5

print("\n=== STAGE 2: Representational Drift ===")
prev_loss = 0.0
start_drift_time = time.time()
for epoch in range(1, MAX_DRIFT_EPOCHS + 1):
    print(f"Warm-up Epoch {epoch}/{MAX_WARMUP_EPOCHS}...")
    
    # Run your function
    train_loss, test_loss = train_and_eval_epoch(epoch=epoch, model=model, mp_train_loader=mp_train_loader,
                                                       mp_test_loader=mp_test_loader,optimizer=optimizer, loss_fn=loss_fn,
                                                       device=device, args=args)
    
    train_loss_list_drift.append(train_loss)
    test_loss_list_drift.append(test_loss)
    loss_diff = abs(prev_loss - test_loss)
    # THE BREAK CONDITION
    if loss_diff <= CONFIDENCE_THRESHOLD:
        print(f"\n>>> THRESHOLDS MET at Epoch {epoch} <<<")
        print(">>> Head is fully calibrated. Breaking warm-up cycle.")
        save_checkpoint(model=model, optimizer=optimizer, epoch=epoch, loss=test_loss_list_drift[-1]) 
        break # Exit the warm-up loop early

    if epoch == MAX_WARMUP_EPOCHS:
        save_checkpoint(model=model, optimizer=optimizer, epoch=epoch, loss=test_loss_list_drift[-1])       
    prev_loss = test_loss

print(f"Finished stage 2 in {time.time() - start_drift_time}")


=== STAGE 2: Representational Drift ===
Warm-up Epoch 1/20...
Train Epoch: 1 [0/1408 (0%)]	Loss: 1.912680
Train Epoch: 1 [384/1408 (27%)]	Loss: 2.042995
Train Epoch: 1 [768/1408 (55%)]	Loss: 2.034754
Train Epoch: 1 [1152/1408 (82%)]	Loss: 1.945141
--- Epoch 1 Training Metrics ---
Average Train Loss: 1.988244
Total time of epoch 1 is 50.57s

Warm-up Epoch 2/20...
Train Epoch: 2 [0/1408 (0%)]	Loss: 1.920604
Train Epoch: 2 [384/1408 (27%)]	Loss: 2.014866
Train Epoch: 2 [768/1408 (55%)]	Loss: 1.922987
Train Epoch: 2 [1152/1408 (82%)]	Loss: 1.881750
--- Epoch 2 Training Metrics ---
Average Train Loss: 1.937743
Total time of epoch 2 is 26.40s

Warm-up Epoch 3/20...
Train Epoch: 3 [0/1408 (0%)]	Loss: 1.929365
Train Epoch: 3 [384/1408 (27%)]	Loss: 1.867396
Train Epoch: 3 [768/1408 (55%)]	Loss: 1.875807
Train Epoch: 3 [1152/1408 (82%)]	Loss: 1.889117
--- Epoch 3 Training Metrics ---
Average Train Loss: 1.920910
Total time of epoch 3 is 27.35s

Warm-up Epoch 4/20...
Train Epoch: 4 [0/1408 (0%)]